[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dongzoolee/hidden-bites/blob/main/notebooks/review-expression-analysis.ipynb)

# 네이버 지도 리뷰 표현 분석

이 노트북은 `datasets/naver-map-reviews-2026-05-13.json`에 들어 있는 네이버 지도 리뷰 500개를 바탕으로, 유저들이 실제로 사용한 표현을 추출하고 비슷한 의미의 표현끼리 묶어 봅니다.

수업에서 배운 Word2Vec은 한 단어가 주변 단어와 함께 나타나는 패턴을 이용해 의미를 벡터로 배웁니다. 이 프로젝트의 corpus는 아직 500개 리뷰로 작기 때문에 Word2Vec을 처음부터 학습하면 안정적인 임베딩을 얻기 어렵습니다. 대신 같은 아이디어를 더 작은 데이터에 맞게 바꿉니다. 여기서는 표현이 어떤 리뷰, 장소, 카테고리, 네이버 키워드, 방문 맥락과 함께 등장했는지를 co-occurrence vector로 만들고, cosine similarity로 가까운 표현들을 묶습니다.

목표는 두 가지입니다. 첫째, `맛있어요`, `친절해요`, `분위기`처럼 여러 리뷰에 넓게 퍼진 dense한 표현군을 찾습니다. 둘째, 빈도는 낮지만 특정 장소를 잘 설명하는 sparse한 표현을 찾아 Hidden Bites의 장소별 signature로 사용할 수 있게 합니다.

## 0. 실행 환경 준비

Colab에서 바로 실행할 수 있도록 필요한 패키지를 설치합니다. 로컬에서 이미 설치되어 있다면 이 셀은 빠르게 지나갑니다. `kiwipiepy`는 한국어 형태소 분석을 위해 사용하고, `scikit-learn`은 co-occurrence vector와 cosine similarity 계산에 사용합니다. `koreanize-matplotlib`은 차트의 한글 라벨이 깨지지 않도록 사용합니다.

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib networkx kiwipiepy koreanize-matplotlib sentence-transformers ipywidgets plotly


## 1. 데이터 로드와 flat table 구성

원본 JSON은 장소별로 리뷰가 중첩된 구조입니다. 분석하기 쉽도록 리뷰 한 개가 한 행이 되도록 펼칩니다. 원문 리뷰, 네이버 키워드, 장소명, 카테고리, 방문 목적, 동행 형태, 대기 시간 정보를 모두 보존합니다.

In [ ]:
import json
import re
from collections import Counter
from urllib.request import urlopen

try:
    import koreanize_matplotlib
except ModuleNotFoundError:
    pass

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import ipywidgets as widgets
import matplotlib.pyplot as plt
from matplotlib import font_manager
import networkx as nx
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display
from kiwipiepy import Kiwi
from scipy.sparse import csr_matrix, hstack
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    import google.colab
    pio.renderers.default = "colab"
except ModuleNotFoundError:
    pio.renderers.default = "notebook_connected"

dataset_url = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/naver-map-reviews-2026-05-13.json"

with urlopen(dataset_url) as response:
    data = json.loads(response.read().decode("utf-8"))

rows = []

for location_item in data["locations"]:
    location = location_item["location"]
    for review in location_item["reviews"]:
        rows.append(
            {
                "review_id": f"{location_item['position']:02d}-{review['rank']:02d}",
                "location_name": location["name"],
                "location_type": location["type"],
                "category": location.get("category"),
                "rank": review["rank"],
                "visited_date": review.get("visited_date"),
                "visit_time_label": review.get("visit_time_label"),
                "reserved": review.get("reserved"),
                "waiting_label": (review.get("waiting") or {}).get("label"),
                "purpose": review.get("purpose") or [],
                "companion": review.get("companion"),
                "text": review.get("text") or "",
                "naver_keywords": review.get("naver_keywords") or [],
            }
        )

reviews_df = pd.DataFrame(rows)
summary_df = pd.DataFrame(
    [
        {"metric": "locations", "value": data["locations_count"]},
        {"metric": "reviews", "value": data["reviews_count"]},
        {"metric": "non_empty_text_reviews", "value": int(reviews_df["text"].str.strip().ne("").sum())},
    ]
)

display(summary_df)
display(reviews_df.head())


## 2. 분석 기준 설정

여기서 정하는 threshold는 이번 노트북의 해석 규칙입니다. 리뷰 수가 늘어나면 기준도 다시 조정할 수 있습니다.

- `dense`는 여러 리뷰와 여러 장소에서 반복되고, 주변에 유사 표현도 있는 표현입니다.
- `sparse_meaningful`은 빈도는 낮지만 특정 장소에 강하게 몰린 표현입니다.
- `sparse_noise`는 한 번만 등장하고 의미 연결도 약한 표현입니다.

Word2Vec에서 주변 단어 window를 쓰듯이, 이 노트북은 리뷰 하나를 작은 context window로 봅니다. 같은 리뷰 안에서 함께 등장한 키워드, 장소, 카테고리, 방문 맥락이 표현의 의미를 설명하는 feature가 됩니다.

In [ ]:
MIN_TOKEN_LENGTH = 2
MIN_DENSE_REVIEW_COUNT = 8
MIN_DENSE_LOCATION_COUNT = 3
MIN_DENSE_NEIGHBORS = 2
SPARSE_MAX_REVIEW_COUNT = 2
SPARSE_LIFT_THRESHOLD = 2.0
SIMILARITY_THRESHOLD = 0.90
MAX_EXPRESSIONS_FOR_SIMILARITY = 1200
MAX_DENSE_CONTEXT_EXPRESSIONS = 450
MAX_SPARSE_CONTEXT_EXPRESSIONS = 650
MAX_KEYWORD_CONTEXT_EXPRESSIONS = 650
MAX_EDGES_PER_EXPRESSION = 12
MIN_HYBRID_EDGE_SIMILARITY = 0.65
CONTEXT_SIMILARITY_WEIGHT = 0.55
SEMANTIC_SIMILARITY_WEIGHT = 0.45
DEFAULT_SPARSE_MAX_REVIEW_COUNT = 20
DEFAULT_SPARSE_MIN_LOCATION_LIFT = 2.0
DEFAULT_TOP_N = 30
DEFAULT_NETWORK_MAX_NODES = 120
MAX_NETWORK_NODES = 150
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-small"
EMBEDDING_BATCH_SIZE = 64

ALLOWED_TAG_PREFIXES = ("N", "V", "XR", "SL", "SN")

STOPWORDS = {
    "하다",
    "되다",
    "이다",
    "아니다",
    "있다",
    "없다",
    "같다",
    "오다",
    "가다",
    "먹다",
    "보다",
    "싶다",
    "주다",
    "않다",
    "나다",
    "알다",
    "시키다",
    "들다",
    "정도",
    "오늘",
    "이번",
    "저번",
    "방문",
    "이용",
    "매장",
    "곳",
    "때",
    "것",
    "수",
    "듯",
    "진짜",
    "너무",
    "정말",
    "항상",
    "자주",
    "다시",
    "처음",
    "완전",
    "그리고",
    "근데",
    "여기",
    "저희",
    "제가",
}

def resolve_korean_font_family():
    candidate_names = ["NanumGothic", "AppleGothic", "Malgun Gothic", "Noto Sans CJK KR", "Noto Sans KR"]
    available_names = {font.name for font in font_manager.fontManager.ttflist}
    for name in candidate_names:
        if name in available_names:
            return name
    current_family = plt.rcParams.get("font.family", ["sans-serif"])
    if isinstance(current_family, str):
        return current_family
    return current_family[0] if current_family else "sans-serif"


KOREAN_FONT_FAMILY = resolve_korean_font_family()
kiwi = Kiwi()
plt.rcParams["font.family"] = KOREAN_FONT_FAMILY
plt.rcParams["axes.unicode_minus"] = False


## 3. 표현 후보 추출

네이버 키워드는 이미 리뷰 플랫폼이 제공하는 의미 단위이므로 anchor expression으로 그대로 사용합니다. 리뷰 원문에서는 Kiwi 형태소 분석으로 명사, 동사, 형용사, 어근 중심의 normalized token을 뽑습니다.

예를 들어 `맛있고`, `맛있어요`, `맛있습니다`처럼 표면형은 다르지만 의미가 가까운 표현은 `맛있다` 같은 정규화된 token으로 모이게 됩니다. 반대로 원문 전체 문장은 대표 예시로 남겨 나중에 해석할 때 실제 표현을 다시 볼 수 있게 합니다.

In [ ]:
def normalize_text_token(token):
    form = token.form.strip().lower()
    tag = token.tag
    if not form:
        return None
    if not tag.startswith(ALLOWED_TAG_PREFIXES):
        return None
    if tag.startswith("V") and not form.endswith("다"):
        form = f"{form}다"
    if len(form) < MIN_TOKEN_LENGTH:
        return None
    if form in STOPWORDS:
        return None
    if re.fullmatch(r"[0-9]+", form):
        return None
    return form


def normalize_keyword(keyword):
    expression = re.sub(r"\s+", " ", str(keyword).strip())
    if len(expression) < MIN_TOKEN_LENGTH:
        return None
    return expression


def extract_text_expressions(text):
    expressions = []
    for token in kiwi.tokenize(text):
        normalized = normalize_text_token(token)
        if normalized is not None:
            expressions.append(normalized)
    return expressions


expression_rows = []

for row in reviews_df.itertuples(index=False):
    text_expressions = extract_text_expressions(row.text)
    for expression in text_expressions:
        expression_rows.append(
            {
                "review_id": row.review_id,
                "expression": expression,
                "source": "text_token",
            }
        )
    for keyword in row.naver_keywords:
        expression = normalize_keyword(keyword)
        if expression is not None:
            expression_rows.append(
                {
                    "review_id": row.review_id,
                    "expression": expression,
                    "source": "naver_keyword",
                }
            )

expressions_df = pd.DataFrame(expression_rows).drop_duplicates()
expression_review_df = expressions_df.drop_duplicates(["review_id", "expression"])
expression_context_df = expression_review_df.merge(reviews_df, on="review_id", how="left")

source_types_df = (
    expressions_df.groupby("expression")["source"]
    .apply(lambda values: ", ".join(sorted(set(values))))
    .reset_index(name="source_types")
)

example_text_df = (
    expression_context_df[expression_context_df["text"].str.strip().ne("")]
    .groupby("expression")["text"]
    .first()
    .reset_index(name="example_text")
)

expression_stats = (
    expression_context_df.groupby("expression")
    .agg(
        review_count=("review_id", "nunique"),
        location_count=("location_name", "nunique"),
        category_count=("category", "nunique"),
    )
    .reset_index()
    .merge(source_types_df, on="expression", how="left")
    .merge(example_text_df, on="expression", how="left")
    .sort_values(["review_count", "location_count", "expression"], ascending=[False, False, True])
    .reset_index(drop=True)
)

display(expression_stats.head(30))
print(f"표현 후보 수: {len(expression_stats):,}")

,expression,review_count,location_count,category_count,source_types,example_text
0,음식이 맛있어요,302,10,9,naver_keyword,믿고먹는 탕화쿵푸!! 역시 맛있는데 매장도 청결하고 재료도 잘관리돼는거같아요 맛있게...
1,맛있다,266,10,9,text_token,믿고먹는 탕화쿵푸!! 역시 맛있는데 매장도 청결하고 재료도 잘관리돼는거같아요 맛있게...
2,재료가 신선해요,188,10,9,naver_keyword,믿고먹는 탕화쿵푸!! 역시 맛있는데 매장도 청결하고 재료도 잘관리돼는거같아요 맛있게...
3,좋다,187,10,9,text_token,항상 잘 애용하고 있어요. 친절하게 아이 입맛에도 맛게 조리해주시고 아이가 좋아해요.
4,친절해요,138,10,9,naver_keyword,오늘은 마라상궈와 꿔바로우를 먹었는데 역시 맛있네요
5,인테리어가 멋져요,118,10,9,naver_keyword,오늘도 맛있게 먹고 갑니다!
6,특별한 메뉴가 있어요,102,10,9,naver_keyword,압구정에 새로 생겨서 너무 좋아요! 직원 분들도 친절 하고 맛있습니다
7,빵이 맛있어요,92,2,2,naver_keyword,베이글이 맛있고 분위기도 좋아요:) 적극 추천하는 맛집!!
8,양이 많아요,85,8,7,naver_keyword,믿고먹는 탕화쿵푸!! 역시 맛있는데 매장도 청결하고 재료도 잘관리돼는거같아요 맛있게...
9,친절,77,10,9,text_token,아이가 제일 좋아하는 마라탕집이예요. 다른곳도 다녀봤지만 식재료도 깔끔하고 식기류도...


표현 후보 수: 1,127


## 4. Co-occurrence vector 만들기

각 표현을 숫자 벡터로 만들기 위해, 표현이 등장한 리뷰의 주변 정보를 feature로 모읍니다. 이 방식은 작은 데이터셋에서 Word2Vec을 직접 학습하는 대신, Word2Vec의 핵심 직관인 `같은 맥락에 등장하는 표현은 의미가 가깝다`를 직접 구현한 것입니다.

예를 들어 `혼밥하기 좋아요`와 `혼자`가 같은 리뷰, 같은 장소, 같은 키워드와 자주 연결되면 두 표현의 vector가 가까워집니다. `빵이 맛있어요`와 `베이글`도 베이커리/카페 리뷰 맥락에서 가까워질 수 있습니다.

In [ ]:
candidate_pool_df = expression_stats[
    (expression_stats["review_count"] >= 2)
    | expression_stats["source_types"].str.contains("naver_keyword", regex=False)
].copy()

dense_candidate_df = candidate_pool_df.sort_values(
    ["review_count", "location_count", "expression"], ascending=[False, False, True]
).head(MAX_DENSE_CONTEXT_EXPRESSIONS)

sparse_candidate_df = candidate_pool_df[candidate_pool_df["review_count"] <= DEFAULT_SPARSE_MAX_REVIEW_COUNT].sort_values(
    ["review_count", "location_count", "expression"], ascending=[True, False, True]
).head(MAX_SPARSE_CONTEXT_EXPRESSIONS)

keyword_candidate_df = candidate_pool_df[candidate_pool_df["source_types"].str.contains("naver_keyword", regex=False)].sort_values(
    ["review_count", "location_count", "expression"], ascending=[False, False, True]
).head(MAX_KEYWORD_CONTEXT_EXPRESSIONS)

candidate_expressions = (
    pd.concat([sparse_candidate_df, keyword_candidate_df, dense_candidate_df], ignore_index=True)
    .drop_duplicates("expression")
    .head(MAX_EXPRESSIONS_FOR_SIMILARITY)["expression"]
    .tolist()
)

candidate_context_df = expression_context_df[expression_context_df["expression"].isin(candidate_expressions)]

feature_rows = []
feature_expressions = []

for expression, group in candidate_context_df.groupby("expression"):
    features = Counter()
    for row in group.itertuples(index=False):
        features[f"review:{row.review_id}"] += 1.0
        features[f"location:{row.location_name}"] += 1.8
        features[f"category:{row.category}"] += 1.2
        features[f"visit_time:{row.visit_time_label}"] += 0.7
        features[f"companion:{row.companion}"] += 0.7
        features[f"waiting:{row.waiting_label}"] += 0.7
        for purpose in row.purpose:
            features[f"purpose:{purpose}"] += 0.7
        for keyword in row.naver_keywords:
            features[f"keyword:{keyword}"] += 1.4
    feature_expressions.append(expression)
    feature_rows.append(dict(features))

vectorizer = DictVectorizer(sparse=True)
if feature_rows:
    context_matrix = vectorizer.fit_transform(feature_rows)
    similarity_matrix = cosine_similarity(context_matrix)
else:
    context_matrix = csr_matrix((0, 0))
    similarity_matrix = np.empty((0, 0))

expression_index_map = {expression: index for index, expression in enumerate(feature_expressions)}
context_edge_rows = []
G = nx.Graph()
G.add_nodes_from(feature_expressions)

for left_index in range(len(feature_expressions)):
    for right_index in range(left_index + 1, len(feature_expressions)):
        similarity = float(similarity_matrix[left_index, right_index])
        if similarity >= SIMILARITY_THRESHOLD:
            left_expression = feature_expressions[left_index]
            right_expression = feature_expressions[right_index]
            G.add_edge(left_expression, right_expression, weight=similarity)
            context_edge_rows.append(
                {
                    "left_expression": left_expression,
                    "right_expression": right_expression,
                    "context_similarity": similarity,
                }
            )

context_edge_df = pd.DataFrame(context_edge_rows, columns=["left_expression", "right_expression", "context_similarity"])
neighbor_counts = dict(G.degree())

cluster_rows = []
cluster_map = {}

components = sorted(nx.connected_components(G), key=lambda component: (-len(component), sorted(component)[0]))
for cluster_id, component in enumerate(components, start=1):
    component_stats = expression_stats[expression_stats["expression"].isin(component)].copy()
    representative = component_stats.sort_values(["review_count", "location_count", "expression"], ascending=[False, False, True]).iloc[0]
    component_context = expression_context_df[expression_context_df["expression"].isin(component)]
    locations = sorted(component_context["location_name"].dropna().unique().tolist())[:5]
    examples = component_context[component_context["text"].str.strip().ne("")]["text"].drop_duplicates().head(3).tolist()
    for expression in component:
        cluster_map[expression] = cluster_id
    if len(component) >= 2:
        cluster_rows.append(
            {
                "cluster_id": cluster_id,
                "representative": representative["expression"],
                "expression_count": len(component),
                "total_review_mentions": int(component_stats["review_count"].sum()),
                "expressions": ", ".join(component_stats.head(12)["expression"].tolist()),
                "locations": ", ".join(locations),
                "examples": " / ".join(examples),
            }
        )

cluster_columns = ["cluster_id", "representative", "expression_count", "total_review_mentions", "expressions", "locations", "examples"]
if cluster_rows:
    cluster_df = pd.DataFrame(cluster_rows).sort_values(["expression_count", "total_review_mentions"], ascending=[False, False])
else:
    cluster_df = pd.DataFrame(columns=cluster_columns)

display(cluster_df.head(20))
print(f"분석 표현 수: {len(feature_expressions):,}")
print(f"context 유사 표현 edge 수: {G.number_of_edges():,}")
print(f"크기 2 이상 context cluster 수: {len(cluster_df):,}")


## 5. Sparse / dense 표현 분리

표현을 단순히 많이 나온 순서로만 보면 `맛있다`, `좋다`처럼 모든 장소에 흔한 말만 보입니다. Hidden Bites에서는 흔한 만족 표현과 특정 장소를 설명하는 희소 표현을 나눠 보는 것이 중요합니다.

그래서 빈도, 장소 확산도, 유사 이웃 수, 특정 장소 집중도 lift를 함께 봅니다. lift는 어떤 표현이 특정 장소에 몰려 있는 정도입니다. 전체 리뷰 중 한 장소가 차지하는 비율보다, 해당 표현의 리뷰가 그 장소에 나타나는 비율이 훨씬 높으면 lift가 큽니다.

In [ ]:
total_reviews = reviews_df["review_id"].nunique()
location_base_share = reviews_df.groupby("location_name")["review_id"].nunique() / total_reviews
expression_location_counts = expression_context_df.groupby(["expression", "location_name"])["review_id"].nunique()
expression_review_counts = expression_context_df.groupby("expression")["review_id"].nunique()

max_lift_rows = []

for expression, review_count in expression_review_counts.items():
    location_counts = expression_location_counts.loc[expression]
    lift_values = {}
    for location_name, count in location_counts.items():
        expression_share = count / review_count
        lift_values[location_name] = expression_share / location_base_share.loc[location_name]
    top_location = max(lift_values, key=lift_values.get)
    max_lift_rows.append(
        {
            "expression": expression,
            "top_location": top_location,
            "max_location_lift": float(lift_values[top_location]),
        }
    )

lift_df = pd.DataFrame(max_lift_rows)

analysis_stats = expression_stats.merge(lift_df, on="expression", how="left")
analysis_stats["neighbor_count"] = analysis_stats["expression"].map(neighbor_counts).fillna(0).astype(int)
analysis_stats["cluster_id"] = analysis_stats["expression"].map(cluster_map)

if feature_expressions:
    embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    embedding_inputs = [f"query: {expression}" for expression in feature_expressions]
    semantic_vectors = embedding_model.encode(
        embedding_inputs,
        batch_size=EMBEDDING_BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    semantic_vectors = np.asarray(semantic_vectors, dtype=np.float32)
    semantic_similarity_matrix = np.matmul(semantic_vectors, semantic_vectors.T)
    hybrid_similarity_matrix = (CONTEXT_SIMILARITY_WEIGHT * similarity_matrix) + (SEMANTIC_SIMILARITY_WEIGHT * semantic_similarity_matrix)
else:
    semantic_vectors = np.empty((0, 0), dtype=np.float32)
    semantic_similarity_matrix = np.empty((0, 0), dtype=np.float32)
    hybrid_similarity_matrix = np.empty((0, 0), dtype=np.float32)

edge_pair_map = {}

def add_hybrid_edge(left_expression, right_expression):
    left_index = expression_index_map[left_expression]
    right_index = expression_index_map[right_expression]
    ordered_left, ordered_right = sorted([left_expression, right_expression])
    key = (ordered_left, ordered_right)
    edge_pair_map[key] = {
        "left_expression": ordered_left,
        "right_expression": ordered_right,
        "context_similarity": float(similarity_matrix[left_index, right_index]),
        "semantic_similarity": float(semantic_similarity_matrix[left_index, right_index]),
        "hybrid_similarity": float(hybrid_similarity_matrix[left_index, right_index]),
    }

for row in context_edge_df.itertuples(index=False):
    add_hybrid_edge(row.left_expression, row.right_expression)

for left_index, left_expression in enumerate(feature_expressions):
    if len(feature_expressions) < 2:
        continue
    scores = hybrid_similarity_matrix[left_index].copy()
    scores[left_index] = -np.inf
    candidate_count = min(MAX_EDGES_PER_EXPRESSION, len(feature_expressions) - 1)
    nearest_indices = np.argpartition(-scores, candidate_count - 1)[:candidate_count]
    for right_index in nearest_indices:
        right_expression = feature_expressions[int(right_index)]
        context_similarity = float(similarity_matrix[left_index, right_index])
        hybrid_similarity = float(hybrid_similarity_matrix[left_index, right_index])
        if hybrid_similarity >= MIN_HYBRID_EDGE_SIMILARITY or context_similarity >= SIMILARITY_THRESHOLD:
            add_hybrid_edge(left_expression, right_expression)

edge_columns = ["left_expression", "right_expression", "context_similarity", "semantic_similarity", "hybrid_similarity"]
if edge_pair_map:
    edge_df = pd.DataFrame(edge_pair_map.values()).sort_values("hybrid_similarity", ascending=False).reset_index(drop=True)
else:
    edge_df = pd.DataFrame(columns=edge_columns)

if edge_df.empty:
    hybrid_neighbor_counts = pd.Series(dtype=int)
else:
    hybrid_neighbor_counts = pd.concat([edge_df["left_expression"], edge_df["right_expression"]]).value_counts()

semantic_vector_map = {expression: semantic_vectors[index].astype(float).tolist() for index, expression in enumerate(feature_expressions)}
analysis_stats["semantic_vector"] = analysis_stats["expression"].map(semantic_vector_map)
analysis_stats["hybrid_neighbor_count"] = analysis_stats["expression"].map(hybrid_neighbor_counts).fillna(0).astype(int)
analysis_stats["combined_neighbor_count"] = analysis_stats[["neighbor_count", "hybrid_neighbor_count"]].max(axis=1)
analysis_stats["rarity_weight"] = 1 / np.log1p(analysis_stats["review_count"].clip(lower=1))
analysis_stats["sparse_signature_score"] = (
    analysis_stats["max_location_lift"].fillna(0)
    * analysis_stats["rarity_weight"]
    * (1 + analysis_stats["hybrid_neighbor_count"])
)

color_by_label = {"dense": "#2f6f73", "sparse_meaningful": "#c77700", "sparse_noise": "#aaaaaa", "middle": "#6b7280"}

def classify_expression(row):
    has_keyword_anchor = "naver_keyword" in str(row.source_types)
    if (
        row.review_count >= MIN_DENSE_REVIEW_COUNT
        and row.location_count >= MIN_DENSE_LOCATION_COUNT
        and row.combined_neighbor_count >= MIN_DENSE_NEIGHBORS
    ):
        return "dense"
    if (
        row.review_count <= SPARSE_MAX_REVIEW_COUNT
        and row.max_location_lift >= SPARSE_LIFT_THRESHOLD
        and (row.review_count == SPARSE_MAX_REVIEW_COUNT or has_keyword_anchor)
    ):
        return "sparse_meaningful"
    if row.review_count <= 1 and row.combined_neighbor_count == 0:
        return "sparse_noise"
    return "middle"

analysis_stats["density_label"] = analysis_stats.apply(classify_expression, axis=1)

label_summary_df = (
    analysis_stats.groupby("density_label")
    .agg(expressions=("expression", "count"), mean_review_count=("review_count", "mean"), mean_location_count=("location_count", "mean"))
    .reset_index()
    .sort_values("expressions", ascending=False)
)

dense_df = analysis_stats[analysis_stats["density_label"].eq("dense")].sort_values(
    ["review_count", "hybrid_neighbor_count", "location_count"], ascending=[False, False, False]
)

sparse_meaningful_df = analysis_stats[analysis_stats["density_label"].eq("sparse_meaningful")].sort_values(
    ["sparse_signature_score", "max_location_lift", "review_count"], ascending=[False, False, False]
)

sparse_explorer_df = analysis_stats[
    (analysis_stats["review_count"] <= DEFAULT_SPARSE_MAX_REVIEW_COUNT)
    & (analysis_stats["max_location_lift"] >= DEFAULT_SPARSE_MIN_LOCATION_LIFT)
].sort_values(["sparse_signature_score", "max_location_lift", "review_count"], ascending=[False, False, True])

sparse_noise_df = analysis_stats[analysis_stats["density_label"].eq("sparse_noise")].sort_values(
    ["review_count", "expression"], ascending=[False, True]
)

display(label_summary_df)
display(edge_df.head(25))
display(dense_df.head(25)[["expression", "review_count", "location_count", "hybrid_neighbor_count", "source_types", "example_text"]])
display(sparse_explorer_df.head(25)[["expression", "review_count", "top_location", "max_location_lift", "hybrid_neighbor_count", "sparse_signature_score", "example_text"]])
print(f"hybrid edge 후보 수: {len(edge_df):,}")


## 6. 장소별 sparse signature 보기

`sparse_meaningful` 표현은 많이 등장하지 않더라도 특정 장소를 설명하는 단서가 될 수 있습니다. 아래 표는 장소별로 lift가 높은 표현을 모아 보여 줍니다.

이 표는 최종 시각화에서 `이 장소가 다른 장소와 다르게 언급되는 지점`을 만드는 데 사용할 수 있습니다.

In [ ]:
signature_rows = []

for location_name, group in sparse_explorer_df.groupby("top_location"):
    top_group = group.head(12)
    signature_rows.append(
        {
            "location_name": location_name,
            "signature_expressions": ", ".join(top_group["expression"].tolist()),
            "mean_lift": float(top_group["max_location_lift"].mean()),
            "mean_signature_score": float(top_group["sparse_signature_score"].mean()),
            "expression_count": len(top_group),
        }
    )

signature_columns = ["location_name", "signature_expressions", "mean_lift", "mean_signature_score", "expression_count"]
if signature_rows:
    signature_df = pd.DataFrame(signature_rows).sort_values(["mean_signature_score", "mean_lift"], ascending=[False, False])
else:
    signature_df = pd.DataFrame(columns=signature_columns)

display(signature_df)


## 7. Sparse signature bar explorer

아래 explorer는 많이 등장하는 표현 대신 적게 등장하지만 특정 장소에 강하게 몰리는 표현을 찾기 위한 bar chart입니다. `review_count`, 장소 lift, 장소, 표현 source를 조절하면서 sparse signature 후보를 확인합니다.


In [ ]:
location_options = ["전체"] + sorted(analysis_stats["top_location"].dropna().unique().tolist())
source_options = ["전체", "naver_keyword", "text_token"]
density_options = ["전체"] + sorted(analysis_stats["density_label"].dropna().unique().tolist())
sort_options = [
    ("signature score", "sparse_signature_score"),
    ("location lift", "max_location_lift"),
    ("review count", "review_count"),
    ("hybrid neighbors", "hybrid_neighbor_count"),
]

def filter_sparse_candidates(max_review_count, min_location_lift, top_location="전체", source_type="전체", density_label="전체"):
    filtered = analysis_stats[
        (analysis_stats["review_count"] <= max_review_count)
        & (analysis_stats["max_location_lift"] >= min_location_lift)
    ].copy()
    if top_location != "전체":
        filtered = filtered[filtered["top_location"].eq(top_location)]
    if source_type != "전체":
        filtered = filtered[filtered["source_types"].str.contains(source_type, regex=False, na=False)]
    if density_label != "전체":
        filtered = filtered[filtered["density_label"].eq(density_label)]
    return filtered


def plot_sparse_signature_bar(max_review_count, min_location_lift, top_n, top_location, source_type, sort_by):
    filtered = filter_sparse_candidates(max_review_count, min_location_lift, top_location, source_type)
    ascending = sort_by == "review_count"
    sort_columns = [sort_by] + [column for column in ["sparse_signature_score", "max_location_lift"] if column != sort_by]
    sort_orders = [ascending] + [False for _ in sort_columns[1:]]
    plot_data = filtered.sort_values(sort_columns, ascending=sort_orders).head(top_n)
    if plot_data.empty:
        print("조건에 맞는 표현이 없습니다.")
        return
    plot_data = plot_data.sort_values(sort_by, ascending=True)
    fig = px.bar(
        plot_data,
        x=sort_by,
        y="expression",
        orientation="h",
        color="density_label",
        color_discrete_map=color_by_label,
        hover_data={
            "review_count": True,
            "top_location": True,
            "max_location_lift": ":.2f",
            "hybrid_neighbor_count": True,
            "sparse_signature_score": ":.2f",
            "source_types": True,
            "example_text": True,
        },
        title="Sparse signature expressions",
    )
    fig.update_layout(
        height=max(420, 28 * len(plot_data) + 160),
        font_family=KOREAN_FONT_FAMILY,
        xaxis_title=sort_by,
        yaxis_title="표현",
        legend_title_text="density",
        margin={"l": 120, "r": 40, "t": 70, "b": 50},
    )
    fig.show()

bar_controls = {
    "max_review_count": widgets.IntSlider(value=DEFAULT_SPARSE_MAX_REVIEW_COUNT, min=1, max=80, step=1, description="max reviews"),
    "min_location_lift": widgets.FloatSlider(value=DEFAULT_SPARSE_MIN_LOCATION_LIFT, min=1.0, max=15.0, step=0.5, description="min lift"),
    "top_n": widgets.IntSlider(value=DEFAULT_TOP_N, min=5, max=80, step=5, description="top n"),
    "top_location": widgets.Dropdown(options=location_options, value="전체", description="location"),
    "source_type": widgets.Dropdown(options=source_options, value="전체", description="source"),
    "sort_by": widgets.Dropdown(options=sort_options, value="sparse_signature_score", description="sort"),
}
bar_output = widgets.interactive_output(plot_sparse_signature_bar, bar_controls)
display(widgets.VBox(list(bar_controls.values())), bar_output)



## 8. Sparse expression cluster network explorer

노드는 sparse 표현 후보이고, 선은 context similarity와 semantic embedding similarity를 섞은 hybrid similarity가 threshold 이상인 관계입니다. slider는 이미 계산된 `edge_df`와 `analysis_stats`를 filter해서 sparse 표현 쪽으로 포커스를 이동합니다.


In [ ]:
def plot_sparse_network(max_review_count, min_location_lift, min_hybrid_similarity, max_nodes, top_location, density_label):
    filtered_nodes = filter_sparse_candidates(max_review_count, min_location_lift, top_location, "전체", density_label).sort_values(
        ["sparse_signature_score", "max_location_lift", "review_count"], ascending=[False, False, True]
    ).head(max_nodes)
    if filtered_nodes.empty:
        print("조건에 맞는 표현이 없습니다.")
        return
    node_set = set(filtered_nodes["expression"])
    filtered_edges = edge_df[
        edge_df["left_expression"].isin(node_set)
        & edge_df["right_expression"].isin(node_set)
        & (edge_df["hybrid_similarity"] >= min_hybrid_similarity)
    ].copy()
    network_graph = nx.Graph()
    for row in filtered_nodes.itertuples(index=False):
        network_graph.add_node(row.expression)
    for row in filtered_edges.itertuples(index=False):
        network_graph.add_edge(row.left_expression, row.right_expression, weight=row.hybrid_similarity)
    node_stats = filtered_nodes.set_index("expression")
    score_values = node_stats["sparse_signature_score"].fillna(0)
    score_min = float(score_values.min())
    score_max = float(score_values.max())
    score_range = score_max - score_min
    node_sizes = []
    node_colors = []
    for node in network_graph.nodes():
        score = float(node_stats.loc[node, "sparse_signature_score"])
        normalized_score = 0.5 if score_range == 0 else (score - score_min) / score_range
        node_sizes.append(280 + 1800 * normalized_score)
        node_colors.append(color_by_label.get(node_stats.loc[node, "density_label"], "#6b7280"))
    plt.figure(figsize=(14, 10))
    if network_graph.number_of_edges() > 0:
        pos = nx.spring_layout(network_graph, seed=42, k=0.65, weight="weight")
    else:
        pos = nx.spring_layout(network_graph, seed=42, k=0.95)
    edge_widths = [0.6 + 3.0 * max(0, data["weight"] - min_hybrid_similarity) / max(0.01, 1 - min_hybrid_similarity) for _, _, data in network_graph.edges(data=True)]
    nx.draw_networkx_edges(network_graph, pos, alpha=0.28, width=edge_widths, edge_color="#4b5563")
    nx.draw_networkx_nodes(network_graph, pos, node_color=node_colors, node_size=node_sizes, alpha=0.88, linewidths=0.6, edgecolors="#1f2937")
    nx.draw_networkx_labels(network_graph, pos, font_size=9, font_family=KOREAN_FONT_FAMILY)
    plt.title(f"Sparse expression network: {network_graph.number_of_nodes()} nodes, {network_graph.number_of_edges()} edges")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
    display(filtered_nodes[["expression", "review_count", "top_location", "max_location_lift", "hybrid_neighbor_count", "sparse_signature_score", "example_text"]].head(20))

network_controls = {
    "max_review_count": widgets.IntSlider(value=DEFAULT_SPARSE_MAX_REVIEW_COUNT, min=1, max=80, step=1, description="max reviews"),
    "min_location_lift": widgets.FloatSlider(value=DEFAULT_SPARSE_MIN_LOCATION_LIFT, min=1.0, max=15.0, step=0.5, description="min lift"),
    "min_hybrid_similarity": widgets.FloatSlider(value=0.70, min=0.40, max=0.98, step=0.02, description="min sim"),
    "max_nodes": widgets.IntSlider(value=DEFAULT_NETWORK_MAX_NODES, min=20, max=MAX_NETWORK_NODES, step=10, description="max nodes"),
    "top_location": widgets.Dropdown(options=location_options, value="전체", description="location"),
    "density_label": widgets.Dropdown(options=density_options, value="전체", description="density"),
}
network_output = widgets.interactive_output(plot_sparse_network, network_controls)
display(widgets.VBox(list(network_controls.values())), network_output)


## 9. Interactive 2D scatter

표현 vector를 2차원으로 줄여 sparse 후보를 탐색합니다. 기본 좌표는 context feature와 semantic embedding을 합친 hybrid feature SVD이고, dropdown으로 context-only 또는 embedding-only 좌표도 비교할 수 있습니다.


In [ ]:
def build_coordinate_frame(mode):
    if len(feature_expressions) == 0:
        return pd.DataFrame(columns=["expression", "x", "y", "coordinate_mode"])
    if mode == "context SVD":
        matrix = context_matrix
    elif mode == "semantic SVD":
        matrix = csr_matrix(semantic_vectors)
    else:
        matrix = hstack([
            context_matrix * CONTEXT_SIMILARITY_WEIGHT,
            csr_matrix(semantic_vectors * SEMANTIC_SIMILARITY_WEIGHT),
        ])
    coords = TruncatedSVD(n_components=2, random_state=42).fit_transform(matrix)
    frame = pd.DataFrame({"expression": feature_expressions, "x": coords[:, 0], "y": coords[:, 1]})
    frame["coordinate_mode"] = mode
    return frame

coordinate_mode_options = ["hybrid feature SVD", "context SVD", "semantic SVD"]
coordinate_df = pd.concat([build_coordinate_frame(mode) for mode in coordinate_mode_options], ignore_index=True)
scatter_base_df = coordinate_df.merge(
    analysis_stats[
        [
            "expression",
            "review_count",
            "location_count",
            "top_location",
            "max_location_lift",
            "hybrid_neighbor_count",
            "sparse_signature_score",
            "density_label",
            "source_types",
            "example_text",
        ]
    ],
    on="expression",
    how="left",
)


def plot_sparse_scatter(max_review_count, min_location_lift, top_location, density_label, source_type, coordinate_mode):
    filtered = filter_sparse_candidates(max_review_count, min_location_lift, top_location, source_type, density_label)
    plot_data = scatter_base_df[
        scatter_base_df["expression"].isin(filtered["expression"])
        & scatter_base_df["coordinate_mode"].eq(coordinate_mode)
    ].copy()
    if plot_data.empty:
        print("조건에 맞는 표현이 없습니다.")
        return
    plot_data["plot_size"] = plot_data["sparse_signature_score"].clip(lower=0.1)
    fig = px.scatter(
        plot_data,
        x="x",
        y="y",
        size="plot_size",
        color="density_label",
        color_discrete_map=color_by_label,
        hover_name="expression",
        hover_data={
            "review_count": True,
            "location_count": True,
            "top_location": True,
            "max_location_lift": ":.2f",
            "hybrid_neighbor_count": True,
            "sparse_signature_score": ":.2f",
            "source_types": True,
            "example_text": True,
            "plot_size": False,
            "coordinate_mode": False,
        },
        title=f"Sparse expression space: {coordinate_mode}",
    )
    fig.update_layout(
        height=720,
        font_family=KOREAN_FONT_FAMILY,
        xaxis_title="SVD 1",
        yaxis_title="SVD 2",
        legend_title_text="density",
        margin={"l": 40, "r": 40, "t": 70, "b": 50},
    )
    fig.show()

scatter_controls = {
    "max_review_count": widgets.IntSlider(value=DEFAULT_SPARSE_MAX_REVIEW_COUNT, min=1, max=80, step=1, description="max reviews"),
    "min_location_lift": widgets.FloatSlider(value=DEFAULT_SPARSE_MIN_LOCATION_LIFT, min=1.0, max=15.0, step=0.5, description="min lift"),
    "top_location": widgets.Dropdown(options=location_options, value="전체", description="location"),
    "density_label": widgets.Dropdown(options=density_options, value="전체", description="density"),
    "source_type": widgets.Dropdown(options=source_options, value="전체", description="source"),
    "coordinate_mode": widgets.Dropdown(options=coordinate_mode_options, value="hybrid feature SVD", description="coords"),
}
scatter_output = widgets.interactive_output(plot_sparse_scatter, scatter_controls)
display(widgets.VBox(list(scatter_controls.values())), scatter_output)


## 10. 해석 가이드

이 노트북의 결과는 리뷰 텍스트를 해석하기 위한 출발점입니다. `dense` 표현은 전체 리뷰에서 공통적으로 반복되는 평가 언어를 보여 주고, sparse explorer는 적게 등장하지만 특정 장소와 강하게 연결되는 표현을 signature 후보로 보여 줍니다.

주의할 점도 있습니다. 현재 데이터는 최신순 리뷰 표본이므로, 계절성이나 최근 이벤트의 영향을 받을 수 있습니다. 또한 네이버 키워드는 사용자가 직접 선택한 structured signal이라 원문 텍스트보다 더 안정적이지만, 플랫폼이 제공한 선택지의 편향도 포함합니다.

Hybrid similarity는 co-occurrence 맥락과 사전학습 embedding 의미 유사도를 섞은 탐색용 점수입니다. 최종 시각화에서는 표현 cluster를 그대로 정답처럼 쓰기보다, 실제 리뷰 예시와 함께 확인하는 것이 좋습니다. Hidden Bites의 핵심 질문은 `리뷰 수가 많은 곳이 정말 좋은 곳인가?`이므로, 흔한 칭찬 표현뿐 아니라 특정 상황에서만 강하게 드러나는 표현을 같이 보여 주는 방향이 적합합니다.
